# 1. Producing the data  
In this task, we will implement Apache Kafka producers to simulate real-time data streaming. Spark and parallel data processing should not be used in this section, as we are simulating sensors that often lack processing capabilities.  

1. Every 1 second, load 50-100 accident records from the CSV file. You should keep a pointer in the file reading process and advance it per read. The data should be read in chronological order; just the number of records you read every second is a random number between 50 and 100 (inclusive).
2. Add the current timestamp (accident_ts) to the records. 
3. Send your batch of accident data to a Kafka topic with an appropriate name.

In [ ]:
# import libaries
from time import sleep
from json import dumps
from kafka import KafkaProducer
import random
import datetime as dt
import csv

In [ ]:
# configurations
hostip = "kafka"   
topic = "accident_stream"  
csv_file = "streaming_collision.csv"

In [ ]:
#Read the CSV file and store each row as a dictionary
def read_csv(file_name):
    records = []
    with open(file_name, "rt", encoding="utf-8") as file:
        # Read each row as a dictionary
        reader = csv.DictReader(file)
        for row in reader:
            records.append(row)

    return records

In [ ]:
#Create and connect a Kafka producer.
def connect_kafka_producer():
    producer = None
    try:
        producer = KafkaProducer(
            bootstrap_servers=[f"{hostip}:9092"], # Kafka server address
            value_serializer=lambda x: dumps(x).encode("utf-8"),# Convert data to JSON bytes
            key_serializer=lambda x: str(x).encode("utf-8"), # Convert key to bytes
            api_version=(3, 9)
        )
        print("Kafka producer connected successfully.")

    except Exception as ex:
        print("Exception while connecting Kafka.")
        print(str(ex))

    finally:
        return producer

In [ ]:
#Publish one accident record to Kafka.
def publish_message(producer_instance, topic_name, key, value):
    try:
        producer_instance.send(topic_name, key=key, value=value)

    except Exception as ex:
        print("Exception in publishing message.")
        print(str(ex))

In [ ]:
# Add current timestamp to the accident record
def add_accident_timestamp(record):
    record["accident_ts"] = int(dt.datetime.now().timestamp())
    return record

In [ ]:
if __name__ == "__main__":
    print("Sending accident records to Kafka")
    producer = connect_kafka_producer()
    data = read_csv(csv_file)

    # Pointer keeps track of where we are in the CSV file
    pointer = 0
    total_records = len(data)
    batch_id = 0

    while True:
        # Choose a random batch size between 50 and 100
        batch_size = random.randint(50, 100)

        # Restart from the beginning if all records are used
        if pointer >= total_records:
            pointer = 0

        # Read the next batch
        batch = data[pointer:pointer + batch_size]

        # If not enough records remain, continue reading from the start
        if len(batch) < batch_size:
            remaining = batch_size - len(batch)
            batch.extend(data[0:remaining])
            pointer = remaining
        else:
            pointer += batch_size

        # Add current accident_ts to each record
        accident_batch = []
        for row in batch:
            record = row.copy()
            record = add_accident_timestamp(record)
            accident_batch.append(record)

        # Send each record in the batch to Kafka
        for record in accident_batch:
            key = record["collision_index"]
            publish_message(producer, topic, key, record)

        # Ensure all messages are delivered
        producer.flush()

        print(
            f"Batch {batch_id}: sent {len(accident_batch)} records "
            f"to topic '{topic}'. Current pointer = {pointer}"
        )

        batch_id += 1

        # Send one batch every second
        sleep(1)